In [1]:
#imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             roc_curve)
import joblib
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load the scaled features and labels
X_train = pd.read_csv('../data/processed/X_train_features.csv')
X_test = pd.read_csv('../data/processed/X_test_features.csv')
y_train = pd.read_csv('/content/y_train.csv').squeeze("columns")
y_test = pd.read_csv('/content/y_test.csv').squeeze("columns")

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

FileNotFoundError: [Errno 2] No such file or directory: '../data/processed/X_train_features.csv'

In [ ]:
#Set random seed for reproducibility
RANDOM_STATE = 42

In [ ]:
#Logistic Regression 
lr = LogisticRegression(class_weight='balanced', random_state=RANDOM_STATE, max_iter=1000)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)
y_proba_lr = lr.predict_proba(X_test)[:, 1]

In [ ]:
#Random Forest
rf = RandomForestClassifier(class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)
y_proba_rf = rf.predict_proba(X_test)[:, 1]

In [ ]:
#XGBoost
xgb = XGBClassifier(scale_pos_weight=(len(y_train[y_train==0])/len(y_train[y_train==1])),
                    random_state=RANDOM_STATE, use_label_encoder=False, eval_metric='logloss')
xgb.fit(X_train, y_train)
y_pred_xgb = xgb.predict(X_test)
y_proba_xgb = xgb.predict_proba(X_test)[:, 1]

In [ ]:
#Hyperparameter tuning for XGBoost
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 6, 9],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0]
}

xgb_base = XGBClassifier(scale_pos_weight=(len(y_train[y_train==0])/len(y_train[y_train==1])),
                         random_state=RANDOM_STATE, use_label_encoder=False, eval_metric='logloss')

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
grid = GridSearchCV(xgb_base, param_grid, cv=cv, scoring='f1', n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)

print("Best parameters:", grid.best_params_)
print("Best CV F1:", grid.best_score_)

best_xgb = grid.best_estimator_
y_pred_best = best_xgb.predict(X_test)
y_proba_best = best_xgb.predict_proba(X_test)[:, 1]

Fitting 5 folds for each of 72 candidates, totalling 360 fits


KeyboardInterrupt: 

In [ ]:
#Save the best XGBoost model
joblib.dump(best_xgb, '../models/model.pkl')
print("Model saved.")

In [ ]:
#Model Evaluation
#Evaluation Function
def evaluate_model(name, y_true, y_pred, y_proba):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    roc = roc_auc_score(y_true, y_proba)
    return pd.Series([acc, prec, rec, f1, roc],
                     index=['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC'],
                     name=name)

results = pd.DataFrame({
    'Logistic Regression': evaluate_model('LR', y_test, y_pred_lr, y_proba_lr),
    'Random Forest': evaluate_model('RF', y_test, y_pred_rf, y_proba_rf),
    'XGBoost (tuned)': evaluate_model('XGB', y_test, y_pred_best, y_proba_best)
}).T

print(results)


In [ ]:
#model comparison table
results.style.highlight_max(axis=0)


In [ ]:
#Confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(15,4))
for ax, (name, y_pred) in zip(axes, [('LR', y_pred_lr), ('RF', y_pred_rf), ('XGB', y_pred_best)]):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Legit', 'Phish'], yticklabels=['Legit', 'Phish'])
    ax.set_title(f'{name} Confusion Matrix')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
plt.tight_layout()
plt.savefig('../plots/confusion_matrices.png', dpi=150)
plt.show()

In [ ]:
#ROC curves
plt.figure(figsize=(8,6))
for name, y_proba in [('LR', y_proba_lr), ('RF', y_proba_rf), ('XGB', y_proba_best)]:
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    plt.plot(fpr, tpr, label=f'{name} (AUC = {roc_auc_score(y_test, y_proba):.3f})')

plt.plot([0,1],[0,1], 'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves')
plt.legend()
plt.savefig('../plots/roc_curves.png', dpi=150)
plt.show()


In [ ]:
#Feature importance
importances = best_xgb.feature_importances_
feat_names = X_train.columns
feat_imp = pd.Series(importances, index=feat_names).sort_values(ascending=False)

plt.figure(figsize=(10,6))
feat_imp.plot(kind='bar')
plt.title('XGBoost Feature Importance')
plt.tight_layout()
plt.savefig('../plots/feature_importance.png', dpi=150)
plt.show()